# Chapter 8 — Charts and Maps that Tell the Story

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch08_charts_that_tell_the_story.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 8.1 Gross value added by industry

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import matplotlib.colors as mcolors

from indiadata import mospi_nas, read_plfs, labour_status, labour_rates

gva = mospi_nas("GVA")
top = gva[gva["subindustry"].isna() & gva["institutional_sector"].isna()]
counts = top.pivot_table(index="industry", columns="fy", values="constant_price",
                         aggfunc="count")
counts.index = counts.index.str[:48]                 # shorter labels to print
counts.loc[:, [2013, 2014, 2023, 2024]].fillna(0).astype(int)

In [ ]:
def eight_groups(prices: str) -> pd.DataFrame:
    """GVA in eight consistent industry groups, fiscal years in rows."""
    w = top.pivot(index="fy", columns="industry", values=prices)
    pairs = {
        "Trade, Hotels, Transport, Communication & Services Related to Broadcasting":
            ["Trade, Repair, Hotels and Restaurants",
             "Transport, Storage, Communication & Services Related to Broadcasting"],
        "Financial, Real Estate & Professional Services":
            ["Financial Services",
             "Real Estate, Ownership of Dwelling & Professional Services"],
        "Public Administration, Defence & Other Services":
            ["Public Administration and Defence", "Other Services"],
    }
    out = pd.DataFrame({
        "Agriculture": w["Agriculture, Livestock, Forestry and Fishing"],
        "Mining": w["Mining and Quarrying"],
        "Manufacturing": w["Manufacturing"],
        "Utilities": w["Electricity, Gas, Water Supply & Other Utility Services"],
        "Construction": w["Construction"],
    })
    short = ["Trade, transport, comm.", "Finance, real estate, prof.",
             "Public admin. & other"]
    for name, (combined, parts) in zip(short, pairs.items()):
        out[name] = w[parts].sum(axis=1, min_count=2).fillna(w[combined])
    out["Total"] = w["Total Gross Value Added"]
    return out


real = eight_groups("constant_price")
nominal = eight_groups("current_price")
gap = (real.drop(columns="Total").sum(axis=1) / real["Total"] - 1) * 100
print("Largest gap between sum of groups and total GVA: "
      f"{gap.abs().max():.3f}%")

In [ ]:
SECTOR = {"Agriculture": "Agriculture", "Mining": "Industry",
          "Manufacturing": "Industry", "Utilities": "Industry",
          "Construction": "Industry", "Trade, transport, comm.": "Services",
          "Finance, real estate, prof.": "Services",
          "Public admin. & other": "Services"}


def shares(table: pd.DataFrame) -> pd.DataFrame:
    s = table.drop(columns="Total").T.groupby(SECTOR).sum().T
    return 100 * s.div(table["Total"], axis=0)


share_real, share_nom = shares(real), shares(nominal)
pd.concat({"constant 2011-12 prices": share_real.loc[[2011, 2019, 2025]],
           "current prices": share_nom.loc[[2011, 2019, 2025]]}, axis=1).round(1)

## 8.2 Figure and axes: how matplotlib is organised

In [ ]:
COL = {"Agriculture": "#c55a11", "Industry": "#7f7f7f", "Services": "#1f4e79"}
years = share_real.index


def fy_ticks(ax, first, last, step=2):
    ax.set_xticks(range(first, last + 1, step))
    ax.set_xticklabels([f"{y}–{str(y + 1)[2:]}" for y in range(first, last + 1, step)])


fig, ax = plt.subplots(figsize=(9, 4.8))
for sector, colour in COL.items():
    ax.plot(years, share_real[sector], color=colour, lw=2.4)
    ax.text(years[-1] + 0.25, share_real[sector].iloc[-1],
            f"{sector} {share_real[sector].iloc[-1]:.0f}%",
            color=colour, va="center", fontsize=10, fontweight="bold")
ax.set_ylim(0, 60)
ax.set_xlim(years[0], years[-1] + 2.6)
fy_ticks(ax, years[0], years[-1])
ax.set_ylabel("Share of GVA, % (constant 2011–12 prices)")
ax.set_title("Services' share rose until 2019–20, then levelled off; "
             "agriculture's fell", loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5", lw=0.8)
ax.set_axisbelow(True)
fig.text(0.01, -0.03, "2025–26 is a First Advance Estimate. Source: MoSPI, "
         "National Accounts Statistics (GVA by industry, base 2011–12).",
         fontsize=8, color="grey")
plt.show()

## 8.3 Parts of a whole

In [ ]:
eight_nom = 100 * nominal.drop(columns="Total").div(nominal["Total"], axis=0)
print(f"Manufacturing share at current prices: "
      f"{eight_nom.loc[2011, 'Manufacturing']:.1f}% (2011-12), "
      f"{eight_nom.loc[2025, 'Manufacturing']:.1f}% (2025-26)")

## 8.4 Annotating an event: COVID-19

In [ ]:
real3 = real.drop(columns="Total").T.groupby(SECTOR).sum().T
growth3 = 100 * real3.pct_change()

fig, ax = plt.subplots(figsize=(9, 4.8))
for sector, colour in COL.items():
    ax.plot(growth3.index, growth3[sector], color=colour, lw=2.2, marker="o",
            ms=3.5)
    ax.text(growth3.index[-1] + 0.2, growth3[sector].iloc[-1], sector,
            color=colour, va="center", fontsize=10, fontweight="bold")
ax.axhline(0, color="black", lw=0.8)
worst = growth3.loc[2020].idxmin()
ax.annotate(f"COVID-19 lockdown, 2020–21:\n{worst.lower()} {growth3.loc[2020, worst]:.1f}%",
            xy=(2020, growth3.loc[2020, worst]), xytext=(2014.6, -9),
            arrowprops=dict(arrowstyle="->", color="black", lw=0.8),
            fontsize=9)
ax.annotate(f"agriculture kept growing: +{growth3.loc[2020, 'Agriculture']:.1f}%",
            xy=(2020, growth3.loc[2020, "Agriculture"]), xytext=(2021.3, -3),
            arrowprops=dict(arrowstyle="->", color="#c55a11", lw=0.8),
            fontsize=9, color="#c55a11")
ax.set_xlim(2012, growth3.index[-1] + 1.6)
fy_ticks(ax, 2012, growth3.index[-1])
ax.set_ylabel("Real GVA growth, %")
ax.set_title("The pandemic hit services hardest; agriculture kept growing",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Constant 2011–12 prices. Source: MoSPI, National "
         "Accounts Statistics (GVA by industry).", fontsize=8, color="grey")
plt.show()

In [ ]:
growth3.loc[[2019, 2020, 2021]].round(1)

## 8.5 Who works where? Employment by industry in the PLFS

In [ ]:
def nic_group(d):
    if d <= 3:
        return "Agriculture"
    if d <= 9 or 35 <= d <= 39:
        return "Mining & utilities"
    if d <= 33:
        return "Manufacturing"
    if d <= 43:
        return "Construction"
    return "Services"


HAVE_PLFS = Path("data/plfs/perv1.csv").exists()
if HAVE_PLFS:
    p = read_plfs("perv1", columns=["sex", "age", "pas", "sas", "ind_pas", "ind_sas"])
    p["status"] = labour_status(p["pas"], p["sas"])
    a = p[p["age"] >= 15]
    workers = a[a["status"] == "employed"]
    # a missing pas would give <NA>; fillna(False) makes it "not principal"
    principal_work = workers["pas"].between(11, 51).fillna(False)
    nic = workers["ind_pas"].where(principal_work, workers["ind_sas"])
    division = nic // 1000
    print("workers in sample:", len(workers), "| missing industry code:",
          int(division.isna().sum()))

    workers = workers.assign(group=division.map(nic_group))
    emp_share = (100 * workers.groupby("group")["weight"].sum()
                 / workers["weight"].sum()).sort_values(ascending=False)
    unweighted_agri = 100 * (workers["group"] == "Agriculture").mean()
    print(f"Agriculture's share of workers, unweighted: {unweighted_agri:.1f}%")
    # female LFPR by state, for the maps of section 8.9
    f_state = (a[(a["sex"] == 2).fillna(False)]
               .groupby("st").apply(labour_rates)["LFPR"])
    del p, a, workers, nic, division          # free the microdata
    display(emp_share.round(1))
else:
    f_state = pd.Series(dtype=float)          # empty: maps will show n/a
    print("PLFS files not found in data/plfs/: skipping (see Chapter 14).")

## 8.6 Output versus jobs

In [ ]:
g23 = nominal.loc[2023]
gva_share = pd.Series({
    "Agriculture": g23["Agriculture"],
    "Mining & utilities": g23["Mining"] + g23["Utilities"],
    "Manufacturing": g23["Manufacturing"],
    "Construction": g23["Construction"],
    "Services": g23[["Trade, transport, comm.", "Finance, real estate, prof.",
                     "Public admin. & other"]].sum(),
}) / g23["Total"] * 100
assert abs(gva_share.sum() - 100) < 0.01

if HAVE_PLFS:
    compare = pd.DataFrame({"share of GVA %": gva_share,
                            "share of workers %": emp_share}).loc[gva_share.index]
    compare["relative productivity"] = (compare["share of GVA %"]
                                        / compare["share of workers %"])
    display(compare.round(2))
    agri = compare.loc["Agriculture"]
    print(f"Output per worker in agriculture is {agri['relative productivity']:.2f} "
          f"times the economy average, and "
          f"{agri['relative productivity'] / compare.loc['Services', 'relative productivity']:.2f}"
          f" times that in services")
else:
    print("PLFS files not found: skipping the output-versus-jobs comparison.")

In [ ]:
if HAVE_PLFS:
    order = compare.index[::-1]
    y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    ax.barh(y + 0.2, compare.loc[order, "share of GVA %"], height=0.4,
            color="#1f4e79")
    ax.barh(y - 0.2, compare.loc[order, "share of workers %"], height=0.4,
            color="#c55a11")
    for i, s in enumerate(order):
        ax.text(compare.loc[s, "share of GVA %"] + 0.6, i + 0.2,
                f"{compare.loc[s, 'share of GVA %']:.0f}% of output",
                va="center", fontsize=8.5, color="#1f4e79")
        ax.text(compare.loc[s, "share of workers %"] + 0.6, i - 0.2,
                f"{compare.loc[s, 'share of workers %']:.0f}% of workers",
                va="center", fontsize=8.5, color="#c55a11")
    ax.set_yticks(y)
    ax.set_yticklabels(order)
    ax.set_xlim(0, 72)
    ax.set_xlabel("%")
    ax.set_title("Agriculture: a small share of output, the largest share of jobs",
                 loc="left", fontsize=12)
    ax.spines[["top", "right"]].set_visible(False)
    fig.text(0.01, -0.04, "Output: GVA at current prices, 2023–24 (MoSPI NAS). "
             "Workers: usual status (ps+ss), aged 15+, PLFS July 2023–June 2024,\n"
             "weighted; author's calculation from unit-level data.",
             fontsize=8, color="grey")
    plt.show()

## 8.7 Small multiples: one panel per state

In [ ]:
gsdp = mospi_nas("GSDP")
idx = gsdp.pivot(index="fy", columns="state", values="constant_price").loc[2011:2024]
idx = 100 * idx / idx.iloc[0]
gdp = mospi_nas("GDP").set_index("fy")["constant_price"].loc[2011:2024]
india = 100 * gdp / gdp.iloc[0]
order = idx.iloc[-1].sort_values(ascending=False).index   # fastest first

fig, axes = plt.subplots(5, 6, figsize=(11, 9.5), sharex=True, sharey=True)
for ax, state in zip(axes.flat, order):
    ax.plot(india.index, india, color="#bfbfbf", lw=1.2)
    ax.plot(idx.index, idx[state], color="#1f4e79", lw=1.8)
    ax.axvspan(2019.6, 2020.4, color="#f4d8c4", lw=0)
    name = state.replace("Andaman & Nicobar Islands", "Andaman & Nicobar")
    ax.set_title(f"{name}  {idx[state].iloc[-1]:.0f}", fontsize=8.5, loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(labelsize=7.5)
    ax.set_yticks([100, 200, 300])
    ax.set_xticks([2011, 2017, 2024])
    ax.set_xticklabels(["11–12", "17–18", "24–25"])
fig.suptitle("Real GSDP, 2011–12 = 100: states ordered from fastest to slowest "
             "growth", x=0.01, ha="left", fontsize=13)
fig.text(0.01, 0.005, "Blue: state; grey: all-India GDP; shaded: 2020–21. "
         "Number after the name: index in 2024–25.\nSource: MoSPI, National "
         "Accounts Statistics (GSDP and GDP, constant 2011–12 prices).",
         fontsize=8, color="grey")
fig.tight_layout(rect=(0, 0.04, 1, 0.97))
plt.show()

## 8.8 Colour that works for everyone

## 8.9 A tile-grid map of Indian states

In [ ]:
TILES = {  # state (PLFS layout code): (abbreviation, column, row)
    1: ("JK", 2, 0), 37: ("LA", 3, 0),
    4: ("CH", 1, 1), 3: ("PB", 2, 1), 2: ("HP", 3, 1), 5: ("UK", 4, 1),
    8: ("RJ", 1, 2), 6: ("HR", 2, 2), 7: ("DL", 3, 2), 9: ("UP", 4, 2),
    10: ("BR", 5, 2), 11: ("SK", 6, 2), 18: ("AS", 7, 2), 12: ("AR", 8, 2),
    24: ("GJ", 1, 3), 23: ("MP", 2, 3), 22: ("CG", 3, 3), 20: ("JH", 4, 3),
    19: ("WB", 5, 3), 17: ("ML", 7, 3), 13: ("NL", 8, 3),
    25: ("DD", 1, 4), 27: ("MH", 2, 4), 36: ("TS", 3, 4), 21: ("OD", 4, 4),
    16: ("TR", 7, 4), 14: ("MN", 8, 4),
    30: ("GA", 1, 5), 29: ("KA", 2, 5), 28: ("AP", 3, 5), 15: ("MZ", 8, 5),
    31: ("LD", 0, 6), 32: ("KL", 2, 6), 33: ("TN", 3, 6), 34: ("PY", 4, 6),
    35: ("AN", 6, 6),
}
assert len({(c, r) for _, c, r in TILES.values()}) == len(TILES), "two tiles overlap"
if HAVE_PLFS:
    assert set(TILES) == set(f_state.index), "tile list and PLFS states differ"

In [ ]:
def tile_map(ax, values, cmap, norm, fmt="{:.0f}"):
    """Draw a tile-grid map of Indian states on ax."""
    for code, (abbr, col, row) in TILES.items():
        v = values.get(code, np.nan)
        face = "white" if pd.isna(v) else cmap(norm(v))
        ax.add_patch(plt.Rectangle((col, -row), 0.94, 0.94, facecolor=face,
                                   edgecolor="#bfbfbf" if pd.isna(v) else face,
                                   hatch="////" if pd.isna(v) else None))
        r, g, b, _ = mcolors.to_rgba(face)
        ink = "black" if 0.299 * r + 0.587 * g + 0.114 * b > 0.55 else "white"
        ax.text(col + 0.47, -row + 0.62, abbr, ha="center", va="center",
                fontsize=9, fontweight="bold", color=ink)
        label = "n/a" if pd.isna(v) else fmt.format(v)
        label = label.replace("+0.0", "0.0").replace("-0.0", "0.0")
        ax.text(col + 0.47, -row + 0.28, label, ha="center", va="center",
                fontsize=7.5, color=ink,
                bbox=dict(fc="white", ec="none", pad=0.5) if pd.isna(v) else None)
    ax.set_xlim(-0.1, 9.1)
    ax.set_ylim(-6.1, 1)
    ax.set_aspect("equal")
    ax.axis("off")

In [ ]:
w = gsdp.pivot(index="state", columns="fy", values="constant_price")
cagr = 100 * ((w[2024] / w[2011]) ** (1 / 13) - 1)
india_cagr = 100 * ((gdp[2024] / gdp[2011]) ** (1 / 13) - 1)
NAME_TO_CODE = {   # NAS name -> PLFS code (the crosswalk of Chapter 7)
    "Jammu & Kashmir": 1, "Himachal Pradesh": 2, "Punjab": 3, "Chandigarh": 4,
    "Uttarakhand": 5, "Delhi": 7, "Rajasthan": 8, "Uttar Pradesh": 9,
    "Bihar": 10, "Sikkim": 11, "Arunachal Pradesh": 12, "Manipur": 14,
    "Mizoram": 15, "Tripura": 16, "Meghalaya": 17, "West Bengal": 19,
    "Jharkhand": 20, "Odisha": 21, "Chhattisgarh": 22, "Madhya Pradesh": 23,
    "Gujarat": 24, "Maharashtra": 27, "Andhra Pradesh": 28, "Karnataka": 29,
    "Goa": 30, "Kerala": 32, "Tamil Nadu": 33, "Puducherry": 34,
    "Andaman & Nicobar Islands": 35, "Telangana": 36,
}
assert set(NAME_TO_CODE) == set(cagr.index)
rel_growth = (cagr - india_cagr).rename(index=NAME_TO_CODE)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.6))
cmap1, norm1 = plt.get_cmap("cividis"), mcolors.Normalize(15, 75)
tile_map(ax1, f_state, cmap1, norm1)
ax1.set_title("Female LFPR, 2023–24 (% of women 15+)", loc="left", fontsize=12)
fig.colorbar(plt.cm.ScalarMappable(norm=norm1, cmap=cmap1), ax=ax1,
             orientation="horizontal", fraction=0.04, pad=0.02)

cmap2, norm2 = plt.get_cmap("PuOr"), mcolors.TwoSlopeNorm(0, vmin=-3.5, vmax=3.5)
tile_map(ax2, rel_growth, cmap2, norm2, fmt="{:+.1f}")
ax2.set_title("Real GSDP growth minus India's, 2011–12 to 2024–25 (points)",
              loc="left", fontsize=12)
fig.colorbar(plt.cm.ScalarMappable(norm=norm2, cmap=cmap2), ax=ax2,
             orientation="horizontal", fraction=0.04, pad=0.02)
fig.text(0.01, 0.01, "Tiles are placed approximately; this is not a map of "
         "boundaries. n/a: no GSDP series in MoSPI's table. Sources: author's "
         "calculation from PLFS 2023–24 unit-level data (usual status, weighted);"
         "\nMoSPI, National Accounts Statistics (GSDP and GDP, constant "
         "prices).", fontsize=8, color="grey")
plt.show()

In [ ]:
if HAVE_PLFS:
    print(f"Female LFPR: highest {TILES[f_state.idxmax()][0]} "
          f"({f_state.max():.1f}%), lowest {TILES[f_state.idxmin()][0]} "
          f"({f_state.min():.1f}%)")
print(f"States growing faster than India: {(rel_growth > 0).sum()} of "
      f"{len(rel_growth)}")

## 8.10 Saving figures for publication

In [ ]:
FIGS = Path("notebooks/figures")
FIGS.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGS / "ch08_tile_maps.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGS / "ch08_tile_maps.pdf", bbox_inches="tight")
for f in sorted(FIGS.glob("ch08_*")):
    print(f"{f.name:22s} {f.stat().st_size / 1e3:7.0f} KB")

## 8.11 Common chart mistakes

## 8.12 Answering the question

In [ ]:
print(f"Agriculture's share of real GVA: {share_real.loc[2011, 'Agriculture']:.1f}% "
      f"(2011-12) -> {share_real.loc[2025, 'Agriculture']:.1f}% (2025-26)")
print(f"Services' share of real GVA:     {share_real.loc[2011, 'Services']:.1f}% "
      f"-> {share_real.loc[2025, 'Services']:.1f}%")
print(f"Industry's share of real GVA:    {share_real.loc[2011, 'Industry']:.1f}% "
      f"-> {share_real.loc[2025, 'Industry']:.1f}%")
print(f"Services' share in 2019-20: {share_real.loc[2019, 'Services']:.1f}%")
if HAVE_PLFS:
    print(f"In 2023-24 agriculture had {compare.loc['Agriculture', 'share of GVA %']:.1f}% "
          f"of GVA but {compare.loc['Agriculture', 'share of workers %']:.1f}% of workers")

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading